In [0]:
%run ../utils/__utils__

In [0]:
config = get_config()
catalog = config["catalog"]
tabella_origine = f"{catalog}.bronze.co2_emissions_canada"
ref_carburante  = f"{catalog}.silver.ref_tipo_carburante"
ref_cambio      = f"{catalog}.silver.ref_tipo_cambio"
tabella         = f"{catalog}.silver.veicoli_emissioni"
print(f"Leggo: {tabella_origine}")
print(f"Scrivo in: {tabella}")

In [0]:
def controlla_risultato():
    df = spark.table(tabella)
    print(f"Righe totali: {df.count()}")
    senza_carburante = df.filter("fuel_type IS NULL").count()
    senza_cambio = df.filter("transmission_type IS NULL").count()
    print(f"Righe con carburante non decodificato: {senza_carburante}")
    print(f"Righe con cambio non decodificato: {senza_cambio}")
    display(df.limit(10))

steps_list = [
    {"name": "crea_veicoli_emissioni",
     "query": f"""
        CREATE OR REPLACE TABLE {tabella} AS
        WITH senza_duplicati AS (
            SELECT DISTINCT * EXCEPT (data_caricamento)
            FROM {tabella_origine}
        ),
        cambio_diviso AS (
            SELECT *,
                   regexp_extract(transmission, '^([A-Z]+)', 1)                AS transmission_code,
                   try_cast(regexp_extract(transmission, '([0-9]+)$', 1) AS INT) AS gears
            FROM senza_duplicati
        )
        SELECT
            v.make,
            v.model,
            v.vehicle_class,
            v.engine_size_l,
            v.cylinders,
            c.transmission_type,
            v.gears,
            f.fuel_type,
            v.fuel_consumption_city_l_100_km AS fuel_consumption_city_l_100km,
            v.fuel_consumption_hwy_l_100_km  AS fuel_consumption_hwy_l_100km,
            v.fuel_consumption_comb_l_100_km AS fuel_consumption_comb_l_100km,
            v.co2_emissions_g_km             AS co2_g_km,
            current_timestamp()              AS data_caricamento
        FROM cambio_diviso v
        LEFT JOIN {ref_cambio}     c ON v.transmission_code = c.transmission_code
        LEFT JOIN {ref_carburante} f ON v.fuel_type        = f.fuel_code
     """},
    {"name": "controlla_risultato", "query": controlla_risultato},
]

run_query_send_logs(steps_list)